# Flatfy Listings Analysis

How does data quality affect comparisons of listing prices?

Profile listings, check missing values and duplicates, and compare prices across listing segments.


In [ ]:
import pandas as pd

df = pd.read_csv("data/flatfy_2019_sample.csv.gz")

df.head()

In [ ]:
cols_for_duplicates = df.columns.drop(["Unnamed: 0", "insert_time", "download_time"])
duplicates = df[df.duplicated(subset=cols_for_duplicates, keep=False)]
duplicates.sort_values(by=list(cols_for_duplicates))

n_duplicate_rows_w_insert = df.duplicated(
    subset=list(cols_for_duplicates) + ["insert_time"]
).sum()

n_duplicate_rows_w_download = df.duplicated(
    subset=list(cols_for_duplicates) + ["download_time"]
).sum()

n_duplicate_rows_w_insert_download = df.duplicated(
    subset=list(cols_for_duplicates) + ["insert_time", "download_time"]
).sum()

print(n_duplicate_rows_w_insert)
print(n_duplicate_rows_w_download)
print(n_duplicate_rows_w_insert_download)

In [ ]:
dup_count = df.duplicated(subset=cols_for_duplicates).sum()
dup_count

In [ ]:
df.price.describe() 
df.price.isna().mean() # 0
df['price'].isna().sum() # 0 
df["currency"].value_counts(dropna=False) # USD: 108713, UAH: 33974, EUR: 270
df[df['price'].isna()][['section', 'currency']].value_counts(dropna=False) # No NAN
df.groupby('currency')['price'].describe() # min 1 
df.nlargest(10, 'price')[['price', 'currency', 'section']] # max 2936194460 (a lot of very high values)
df.loc[df['price'] == 1, ['price', 'currency', 'section']] #exist in all currency, and all sections

In [ ]:
(df.price <= 0).sum() # 0 
(df.price <= 1).sum() # 199
(df.price <= 100).sum() # 657
(df.price > 100000000).sum() # 7
df.price.dtype # int64

In [ ]:
df['currency'].dtype #str64
df['currency'].isna().sum() # 0 NAN
df['currency'].isna().mean() # 0 NAN
df['currency'].value_counts(dropna=False) #USD: 108713, UAH: 33974, EUR: 270
df['currency'].nunique(dropna=False) # 3 values

In [ ]:
df['section'].dtype # str64
df['section'].isna().sum() # 90
df['section'].isna().mean() # 0.0006295599376036151
df['section'].value_counts(dropna=False)
df['section'].nunique(dropna=False) #233

In [ ]:
df.room_count.describe() # max 20
df.room_count.dtype # float64
df.room_count.isna().sum() # 20118
df.room_count.isna().mean() # 0.1407276313856614
(df["room_count"] >= 20).sum() # 13

In [ ]:
(df["room_count"] >= 20).sum() # 13

In [ ]:
df.area_total.describe()
df.area_total.max()
df.area_total.dtype # float64
df.area_total.isna().sum() # 13291
df.area_total.isna().mean() # 0.09
(df["area_total"] <= 0).sum() # 0
(df["area_total"] >= 1000).sum() # 1482
(df["area_total"] >= 10000).sum() # 240
df.area_total.nlargest(10,) # needed to be checked after

In [ ]:
df.area_total.nlargest(10,)

In [ ]:
df.geo.describe()
df.geo.dtype # str64
df.geo.isna().sum() # 8
df.geo.isna().mean()
df.geo.value_counts()

In [ ]:
df.geo.value_counts()

In [ ]:
(df['area_living'] > df['area_total']).sum() # 79
(df['area_kitchen'] > df['area_total']).sum() # 49
(df['area_living'] + df['area_kitchen'] > df['area_total']).sum() # 966
(df['floor'] > df['floor_count']).sum() # 112
(df['insert_time'] > df['download_time']).sum() # 0

In [ ]:
(df['insert_time'] > df['download_time']).sum()

In [ ]:
df.section.value_counts()

In [ ]:
listing_cols = df.columns.drop(["Unnamed: 0", "insert_time", "download_time"])

df_no_duplicates = df.drop_duplicates(
    subset=listing_cols,
    keep="first"
)

duplicates_removed = len(df) - len(df_no_duplicates)

count_before_sales_filter = len(df_no_duplicates)

df_sales = df_no_duplicates[
    df_no_duplicates["section"].isin([
        "продаж,квартира",
        "продаж,квартири"
    ])
]

rows_removed_not_sales = count_before_sales_filter - len(df_sales)

df_sales = df_sales[["price", "currency", "room_count"]]

count_before_dropna = len(df_sales)

df_sales = df_sales.dropna(
    subset=["price", "currency", "room_count"]
)

rows_dropped = count_before_dropna - len(df_sales)

usd_mask = df_sales["currency"] == "USD"
uah_mask = df_sales["currency"] == "UAH"
eur_mask = df_sales["currency"] == "EUR"

count_usd = usd_mask.sum()
count_uah = uah_mask.sum()
count_eur = eur_mask.sum()

df_sales["price_usd"] = df_sales["price"].astype(float)

df_sales.loc[uah_mask, "price_usd"] = (
    df_sales.loc[uah_mask, "price"] / 25.85
)

df_sales.loc[eur_mask, "price_usd"] = (
    df_sales.loc[eur_mask, "price"] * 28.95 / 25.85
)

prices_by_rooms = df_sales.groupby("room_count")["price_usd"].agg(
    price_usd="median",
    ad_count="count"
)

print("duplicates_removed:", duplicates_removed)
print("rows_removed_not_sales:", rows_removed_not_sales)
print("rows_dropped:", rows_dropped)
print("count_uah:", count_uah)
print("count_eur:", count_eur)
print("count_usd:", count_usd)

print(prices_by_rooms)